# Notebook 03: Implicit Feedback Models

## Why This Matters

In virtually every real production system, implicit feedback vastly outnumbers explicit feedback. Spotify doesn't ask you to rate every song — it observes what you play, skip, replay, and save. Netflix removed the 5-star rating system in 2017 and replaced it with a thumbs up/down, because implicit engagement signals (completion rate, replay, add-to-list) turned out to be far more predictive. Amazon's purchase and click data dwarfs its review volume by orders of magnitude. Learning to model implicit feedback correctly — especially the fact that *not interacting* is not the same as *disliking* — is the fundamental challenge of production recommender systems.

## Background

### The Core Problem with Implicit Data

Explicit ratings have a natural interpretation: 5 stars = loved it, 1 star = hated it. Implicit data doesn't have this luxury. A user who *didn't* click on an item might:
- Never have seen it (not shown by the system)
- Seen it but wasn't in the right context
- Actually disliked it
- Already own it / already seen it

This is called the **one-class problem**: we have positive observations (interactions), but we don't know which non-interactions are true negatives.

### Hu, Koren & Volinsky (2008): Weighted ALS

The landmark paper "Collaborative Filtering for Implicit Feedback Datasets" introduced a principled solution:

**Confidence**: $c_{ui} = 1 + \alpha \cdot r_{ui}$ where $r_{ui}$ is the raw interaction count (play count, click count). More interactions → higher confidence that the user prefers item $i$. The $\alpha$ hyperparameter (default 40) scales how quickly confidence grows.

**Preference**: $p_{ui} = \mathbb{1}[r_{ui} > 0]$ — binary. Either the user has interacted (preference=1) or not (preference=0).

The model minimizes a *confidence-weighted* loss:
$$\mathcal{L} = \sum_{u,i} c_{ui}(p_{ui} - \mathbf{p}_u \cdot \mathbf{q}_i)^2 + \lambda(\sum_u \|\mathbf{p}_u\|^2 + \sum_i \|\mathbf{q}_i\|^2)$$

Note that the sum is over **all** (u, i) pairs including unobserved ones — but unobserved items have $c_{ui}=1$ (low confidence) and $p_{ui}=0$ (assume dislike). This is computationally expensive ($O(mn)$) but ALS has a trick using the Gramian matrix $Q^T Q$ to make it $O(k^2 n)$ per user.

### Bayesian Personalized Ranking (BPR)

BPR (Rendle et al., 2009) takes a different approach: instead of modeling absolute preference, model **relative preference**. For each user $u$, an observed item $i$ should be ranked higher than an unobserved item $j$.

The BPR loss maximizes the posterior:
$$\mathcal{L}_{BPR} = -\sum_{(u,i,j) \in D_S} \ln \sigma(\hat{x}_{uij}) + \lambda \|\Theta\|^2$$

where $\hat{x}_{uij} = \hat{r}_{ui} - \hat{r}_{uj}$ is the score difference and $D_S$ is a set of (user, positive item, negative item) triples.

**Why BPR works better for ranking**: It directly optimizes the ordering, not the predicted value. A model that correctly ranks positive items above negative items will have high NDCG and Recall@K, even if its absolute score predictions are off.

### Negative Sampling Strategies

The choice of negative items matters enormously:
- **Uniform random**: cheap, easy, but uninformative — model learns to distinguish popular from obscure
- **Popularity-weighted**: sample negatives proportional to $\text{freq}(i)^{0.75}$ — harder negatives, better recall
- **Hard negatives**: items near the decision boundary — high model scores but no interaction — challenging but risky (may be false negatives)

In [ ]:
%pip install -q torch numpy pandas matplotlib tqdm scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, requests, zipfile, io, warnings, time

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "rating", "item_id", "timestamp"],
                      usecols=[0, 1, 2, 3])
# Correct column order
ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1], names=["item_id", "title"])

# Re-index to 0-based
ratings["user_idx"] = ratings.user_id - 1
ratings["item_idx"] = ratings.item_id - 1
n_users = ratings.user_idx.max() + 1
n_items = ratings.item_idx.max() + 1

# Temporal split
ratings = ratings.sort_values(["user_id", "timestamp"])
train_list, test_list = [], []
for uid, group in ratings.groupby("user_id"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])
train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

# Positive item sets per user (for negative sampling & evaluation)
user_pos_train = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
user_pos_test  = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

# Item popularity for weighted negative sampling
item_counts = train_df.item_idx.value_counts().to_dict()
item_pop = np.array([item_counts.get(i, 0) for i in range(n_items)], dtype=np.float32)
item_pop_weighted = item_pop ** 0.75
item_pop_weighted /= item_pop_weighted.sum()

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Device: {device}")
print("Ready.")

## 1. BPR Dataset: Sampling (User, Positive, Negative) Triples

The key to BPR training is how we sample negative items. For each positive (user, item) interaction, we sample one or more negative items the user has *not* interacted with. The model learns to score positive items higher than negative items.

**Popularity-weighted sampling**: sample negatives proportional to $\text{freq}(i)^{0.75}$ — this borrows from word2vec's negative sampling strategy and tends to produce harder, more informative negatives than uniform sampling.

In [ ]:
class BPRDataset(Dataset):
    """
    Samples (user, positive_item, negative_item) triples for BPR training.
    Negative items are sampled with probability proportional to popularity^0.75.
    """

    def __init__(self, train_df, n_items, user_pos_sets, item_pop_weights, n_neg=1):
        self.user_idxs = train_df.user_idx.values
        self.item_idxs = train_df.item_idx.values
        self.n_items = n_items
        self.user_pos = user_pos_sets
        self.pop_w = item_pop_weights
        self.n_neg = n_neg
        self.all_items = np.arange(n_items)

    def __len__(self):
        return len(self.user_idxs) * self.n_neg

    def __getitem__(self, idx):
        base_idx = idx % len(self.user_idxs)
        u = self.user_idxs[base_idx]
        pos_i = self.item_idxs[base_idx]

        # Sample a negative item (not in user's positive set)
        pos_set = self.user_pos.get(u, set())
        for _ in range(50):  # retry up to 50 times
            neg_i = np.random.choice(self.all_items, p=self.pop_w)
            if neg_i not in pos_set:
                break

        return (torch.tensor(u, dtype=torch.long),
                torch.tensor(pos_i, dtype=torch.long),
                torch.tensor(neg_i, dtype=torch.long))


dataset = BPRDataset(train_df, n_items, user_pos_train, item_pop_weighted)
loader  = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=0)
print(f"BPR dataset: {len(dataset):,} triples, {len(loader)} batches/epoch")

## 2. BPR Model: Embedding Tables + Pairwise Loss

The model is simple: user and item embedding tables. The BPR loss maximizes the probability that the positive item scores higher than the negative item:

$$\mathcal{L} = -\frac{1}{|D_S|} \sum_{(u,i,j) \in D_S} \ln \sigma(\hat{r}_{ui} - \hat{r}_{uj}) + \lambda \|\Theta\|^2$$

We add L2 regularization directly in the loss (rather than using the optimizer's weight_decay) so we can control it independently for user and item embeddings.

In [ ]:
class BPRModel(nn.Module):
    """BPR Matrix Factorization with embedding tables."""

    def __init__(self, n_users, n_items, n_factors=64, init_std=0.01):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.normal_(self.user_emb.weight, 0, init_std)
        nn.init.normal_(self.item_emb.weight, 0, init_std)

    def forward(self, users, pos_items, neg_items):
        u  = self.user_emb(users)       # (B, k)
        pi = self.item_emb(pos_items)   # (B, k)
        ni = self.item_emb(neg_items)   # (B, k)

        pos_score = (u * pi).sum(dim=1)  # dot product
        neg_score = (u * ni).sum(dim=1)

        # BPR loss: -log sigmoid(pos_score - neg_score)
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
        return loss, pos_score, neg_score

    def score_all_items(self, user_idx, device):
        """Score all items for a single user — for recommendation."""
        u_emb = self.user_emb.weight[user_idx].unsqueeze(0)  # (1, k)
        scores = (u_emb @ self.item_emb.weight.T).squeeze(0)  # (n_items,)
        return scores.detach().cpu().numpy()


def train_bpr(model, loader, optimizer, reg_lambda=1e-4, n_epochs=20, device="cpu"):
    model.to(device)
    history = {"loss": [], "pos_score": [], "neg_score": []}

    for epoch in range(n_epochs):
        model.train()
        total_loss, total_pos, total_neg, n_batches = 0, 0, 0, 0

        for users, pos_items, neg_items in loader:
            users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
            optimizer.zero_grad()
            loss, ps, ns = model(users, pos_items, neg_items)

            # L2 regularization on embeddings in batch
            u_emb = model.user_emb(users)
            pi_emb = model.item_emb(pos_items)
            ni_emb = model.item_emb(neg_items)
            reg = reg_lambda * (u_emb.norm(2).pow(2) + pi_emb.norm(2).pow(2) + ni_emb.norm(2).pow(2))
            total = loss + reg

            total.backward()
            optimizer.step()

            total_loss += loss.item()
            total_pos += ps.mean().item()
            total_neg += ns.mean().item()
            n_batches += 1

        avg_loss = total_loss / n_batches
        history["loss"].append(avg_loss)
        history["pos_score"].append(total_pos / n_batches)
        history["neg_score"].append(total_neg / n_batches)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | "
                  f"Pos score: {total_pos/n_batches:.3f} | Neg score: {total_neg/n_batches:.3f}")

    return history


model = BPRModel(n_users, n_items, n_factors=64)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Training BPR model (64 factors, 20 epochs)...")
t0 = time.time()
history = train_bpr(model, loader, optimizer, reg_lambda=1e-4, n_epochs=20, device=str(device))
print(f"\nTraining time: {time.time()-t0:.1f}s")

## 3. Evaluation: NDCG@K, Recall@K, MRR

Ranking metrics are the right way to evaluate implicit feedback models — we care about the *order* of items shown to users, not the absolute predicted values.

- **Recall@K**: fraction of the user's actual positive test items that appear in the top-K recommendations. Measures coverage of relevant items.
- **NDCG@K**: Normalized Discounted Cumulative Gain — rewards hitting relevant items earlier in the list. Position-weighted. A hit at rank 1 is worth more than a hit at rank 10.
- **MRR**: Mean Reciprocal Rank of the first relevant item. Useful when you care most about the first result (e.g., search, single-item slots).

$$\text{NDCG@K} = \frac{DCG@K}{IDCG@K}, \quad DCG@K = \sum_{k=1}^{K} \frac{\text{rel}_k}{\log_2(k+1)}$$

In [ ]:
def compute_ranking_metrics(model, user_pos_train, user_pos_test,
                             n_items, K=10, n_eval=300, device="cpu"):
    """Compute Recall@K, NDCG@K, MRR for BPR model."""
    model.eval()
    recalls, ndcgs, mrrs = [], [], []

    eval_users = [u for u in user_pos_test.keys() if len(user_pos_test[u]) > 0]
    eval_users = eval_users[:n_eval]

    with torch.no_grad():
        for u in eval_users:
            test_pos = user_pos_test[u]
            train_pos = user_pos_train.get(u, set())

            # Score all items
            scores = model.score_all_items(u, device)

            # Mask training items
            for i in train_pos:
                scores[i] = -np.inf

            top_k = np.argsort(scores)[::-1][:K]

            # Recall@K
            hits = [1 if item in test_pos else 0 for item in top_k]
            recall = sum(hits) / min(len(test_pos), K)
            recalls.append(recall)

            # NDCG@K
            dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
            ideal_hits = min(len(test_pos), K)
            idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcgs.append(ndcg)

            # MRR
            mrr = 0.0
            for rank, item in enumerate(top_k):
                if item in test_pos:
                    mrr = 1.0 / (rank + 1)
                    break
            mrrs.append(mrr)

    return {
        "Recall@K": np.mean(recalls),
        "NDCG@K": np.mean(ndcgs),
        "MRR": np.mean(mrrs),
        "K": K,
    }


# Evaluate at K=10 and K=20
results_10 = compute_ranking_metrics(model, user_pos_train, user_pos_test, n_items, K=10, device=str(device))
results_20 = compute_ranking_metrics(model, user_pos_train, user_pos_test, n_items, K=20, device=str(device))

print("BPR Model Evaluation:")
for k, res in [("K=10", results_10), ("K=20", results_20)]:
    print(f"  {k}: Recall={res['Recall@K']:.4f}, NDCG={res['NDCG@K']:.4f}, MRR={res['MRR']:.4f}")

## 4. Negative Sampling Strategy Comparison

The choice of negative sampling strategy significantly impacts model quality. Let's compare uniform random sampling vs. popularity-weighted sampling.

In [ ]:
# Compare negative sampling strategies
def make_uniform_pop_weights(n_items):
    return np.ones(n_items) / n_items

uniform_weights = make_uniform_pop_weights(n_items)

results_comparison = {}

for name, weights in [("Uniform", uniform_weights), ("Pop-weighted", item_pop_weighted)]:
    ds = BPRDataset(train_df, n_items, user_pos_train, weights)
    dl = DataLoader(ds, batch_size=1024, shuffle=True, num_workers=0)

    m = BPRModel(n_users, n_items, n_factors=64)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    h = train_bpr(m, dl, opt, reg_lambda=1e-4, n_epochs=15, device=str(device))

    res = compute_ranking_metrics(m, user_pos_train, user_pos_test, n_items, K=10, device=str(device))
    results_comparison[name] = res
    print(f"{name}: Recall@10={res['Recall@K']:.4f}, NDCG@10={res['NDCG@K']:.4f}, MRR={res['MRR']:.4f}")

print("\nPopularity-weighted negatives tend to produce harder training examples.")

## 5. Training Dynamics: Score Separation

A healthy BPR model shows increasing *score separation* between positive and negative items over training. If positive and negative scores converge, the model isn't learning to rank.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history["loss"]) + 1)
axes[0].plot(epochs, history["loss"], color="coral", marker="o", markersize=4)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BPR Loss")
axes[0].set_title("BPR Training Loss")

axes[1].plot(epochs, history["pos_score"], color="steelblue", label="Positive items", marker="s", markersize=4)
axes[1].plot(epochs, history["neg_score"], color="salmon", label="Negative items", marker="^", markersize=4)
axes[1].fill_between(epochs,
                     history["neg_score"],
                     history["pos_score"],
                     alpha=0.15, color="green", label="Score gap")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Score")
axes[1].set_title("Score Separation: Positive vs Negative Items\nGrowing gap = model learning to rank")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/bpr_training.png", dpi=120)
plt.show()
print("The growing score gap between positive and negative items confirms BPR is learning the ranking.")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Implicit feedback | Clicks/views/purchases — abundant but noisy; missing ≠ disliked |
| Hu et al. (2008) | Confidence $c_{ui}=1+\alpha r_{ui}$, preference $p_{ui}=\mathbb{1}[r_{ui}>0]$ — weight unobserved differently |
| BPR (2009) | Pairwise loss: train positive items to score higher than negative items |
| Negative sampling | Choice matters: uniform → easy negatives; pop-weighted → harder, better |
| Hard negatives | High model score + no interaction — informative but risky (false negatives) |
| Recall@K | Fraction of relevant items found in top-K — primary metric for recommendation |
| NDCG@K | Position-weighted precision — rewards hitting relevant items at the top |
| MRR | Mean reciprocal rank of first hit — useful for single-item recommendation slots |

### Common Pitfalls
- Treating implicit feedback like explicit ratings — use pairwise or pointwise with confidence weighting
- Uniform random negative sampling — wastes capacity on trivial negatives
- Evaluating with RMSE on implicit data — it's meaningless; use Recall@K and NDCG@K
- Ignoring false negatives in negative sampling — a "negative" item may have been purchased on a different platform
- Not masking training items during evaluation — inflates Recall@K